# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pr120107/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/pr120107/flyrank-ml-internship.git"
REPO_DIR = "/content/flyrank-ml-internship"

# Clone the repo if it is not already available
if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        check=True
    )

# Move into the repository
os.chdir(REPO_DIR)

print("Working directory:", os.getcwd())
print("Repo contents:", os.listdir()[:10])

Working directory: /content/flyrank-ml-internship
Repo contents: ['DATA_USE.md', 'README.md', 'notebooks', 'docs', 'work', 'SETUP.md', 'data', 'scripts', 'skills', 'outputs']


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [4]:
import pandas as pd
import numpy as np

# Load the same starter dataset used for the Week-4 baseline
df_model = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df_model.shape)

# Create the starter binary target from the observed trend direction
df_model["is_declining_label"] = (
    df_model["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

print("\nTarget distribution:")
print(df_model["is_declining_label"].value_counts())

print("\nTarget proportions:")
print(df_model["is_declining_label"].value_counts(normalize=True))

Dataset shape: (30000, 44)

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Target proportions:
is_declining_label
1    0.542067
0    0.457933
Name: proportion, dtype: float64


## 1. Method choice and why

I chose a Decision Tree Classifier because the target is a binary declining/non-declining outcome and the relationship between content signals may not be purely linear.

A decision tree is also relatively easy to interpret, which makes it useful for understanding which signals the model relies on. This is appropriate for a decision-support use case where the goal is to identify patterns for review rather than claim causality.

The model is intended to support prioritization and review, not to guarantee future performance.

## 2. Split design

I use a stratified 80/20 train-test split so that the proportion of declining and non-declining pages is kept similar in both sets.

A time-aware split would be preferable if reliable observation timestamps were available for this modeling question. The starter dataset does not provide a suitable row-level observation date for that purpose, so I use stratification and treat the test set as a held-out evaluation set.

The test set is kept separate from model fitting and is used only for final evaluation.

In [5]:
from sklearn.model_selection import train_test_split

# Target
y = df_model["is_declining_label"]

# Columns that should not be used as model features
exclude_cols = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

# Candidate features
X = df_model.drop(columns=exclude_cols)

print("Candidate feature count:", X.shape[1])
print("Candidate features:")
print(X.columns.tolist())

# Stratified 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTrain size:", X_train.shape)
print("Test size:", X_test.shape)

print("\nTrain target proportions:")
print(y_train.value_counts(normalize=True))

print("\nTest target proportions:")
print(y_test.value_counts(normalize=True))

Candidate feature count: 40
Candidate features:
['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier']

Train size: (24000, 40)
Test size: (6000, 40)

Train target proportions:
is_declining_label
1    0.542083
0    0.457917
Name: proportion, dtype: float64

Test target proportions:
is_declining_label
1    0.542
0    0.458
Name: proportion, d

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Identify categorical and numeric features
categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

print("Categorical features:", categorical_features)
print("Numeric feature count:", len(numeric_features))

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        ),
        (
            "numeric",
            "passthrough",
            numeric_features
        )
    ]
)

# Decision Tree model
model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)

# Train
pipeline.fit(X_train, y_train)

# Predictions
y_pred = pipeline.predict(X_test)

# ---------------------------------------------------------
# Majority-class baseline
# ---------------------------------------------------------

majority_prediction = np.full(
    len(y_test),
    y_train.mode()[0]
)

baseline_accuracy = accuracy_score(
    y_test,
    majority_prediction
)

baseline_precision = precision_score(
    y_test,
    majority_prediction,
    zero_division=0
)

baseline_recall = recall_score(
    y_test,
    majority_prediction,
    zero_division=0
)

baseline_f1 = f1_score(
    y_test,
    majority_prediction,
    zero_division=0
)

# ---------------------------------------------------------
# Decision Tree evaluation
# ---------------------------------------------------------

model_accuracy = accuracy_score(y_test, y_pred)
model_precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)
model_recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)
model_f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)

comparison = pd.DataFrame({
    "method": [
        "Majority-class baseline",
        "Decision Tree"
    ],
    "accuracy": [
        baseline_accuracy,
        model_accuracy
    ],
    "precision": [
        baseline_precision,
        model_precision
    ],
    "recall": [
        baseline_recall,
        model_recall
    ],
    "f1": [
        baseline_f1,
        model_f1
    ]
})

comparison

Categorical features: ['competition_level', 'content_type', 'main_intent', 'provider_used', 'model_used', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']
Numeric feature count: 29


,method,accuracy,precision,recall,f1
0,Majority-class baseline,0.542000,0.542000,1.000000,0.702983
1,Decision Tree,0.739333,0.720711,0.847478,0.778971


In [7]:
print("Decision Tree classification report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["not_declining", "declining"],
        zero_division=0
    )
)

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

Decision Tree classification report:
               precision    recall  f1-score   support

not_declining       0.77      0.61      0.68      2748
    declining       0.72      0.85      0.78      3252

     accuracy                           0.74      6000
    macro avg       0.75      0.73      0.73      6000
 weighted avg       0.74      0.74      0.73      6000


Confusion matrix:
[[1680 1068]
 [ 496 2756]]


In [9]:
print("Number of original features:", len(X_train.columns))
print("Number of model features:", len(model.feature_importances_))

Number of original features: 40
Number of model features: 77


In [10]:
# ---------------------------------------------------------
# Feature Importance — Top 15
# ---------------------------------------------------------

# Get feature names after preprocessing
feature_names = preprocessor.get_feature_names_out()

# Get importance values from the trained Decision Tree
importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

# Show top 15 features
top15_features = importance_df.head(15)

print("Top 15 transformed features by Decision Tree importance:")
display(top15_features)

Top 15 transformed features by Decision Tree importance:


,feature,importance
0,numeric__impressions_prev_30d,0.600734
1,numeric__impressions_last_30d,0.255567
2,numeric__content_age_days,0.109720
3,numeric__avg_position,0.025226
4,numeric__days_since_last_update,0.008627
5,numeric__cpc,0.000126
6,categorical__content_type_keyword article,0.000000
7,categorical__content_type_feedly article,0.000000
8,categorical__competition_level_HIGH,0.000000
9,categorical__competition_level_LOW,0.000000


### **Interpretation:**
The model's feature importance is strongly concentrated in impressions_prev_30d, impressions_last_30d, and content_age_days, which together account for approximately 96.6% of total tree importance. avg_position and days_since_last_update provide smaller contributions. Most one-hot encoded categorical features have zero importance in this particular tree. These are model-specific associations, not evidence that these variables independently cause content decline. The next validation step will test whether the observed performance and feature relationships remain stable under a more realistic split.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Errors and Interpretation

The Decision Tree achieved 73.93% accuracy and an F1-score of 0.779 on the held-out test set, compared with 54.20% accuracy for the majority-class baseline.

The model correctly identified 2,756 declining pages and 1,680 non-declining pages.

There were 1,068 false positives, where non-declining pages were classified as declining, and 496 false negatives, where declining pages were classified as non-declining.

The higher number of false positives indicates that the model is relatively aggressive in identifying potentially declining pages. However, its 84.75% recall for the declining class means it successfully identifies most declining pages.

The model is therefore useful as a screening or prioritization baseline, but should not be treated as a final decision-maker. Further validation, including feature-importance analysis and leakage checks, is needed before drawing stronger conclusions.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.